# OU Terminal Distribution

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Many OU paths with a live histogram of X_T vs the stationary Gaussian.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/ou-distribution/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults (with many more paths than its 25)
T, dt, theta, mu, sigma, x0 = 8.0, 0.02, 1.2, 0.0, 0.8, 3.0
n_paths = 50000


def exact_moments(t):
    mean = mu + (x0 - mu) * math.exp(-theta * t)
    var = sigma ** 2 / (2 * theta) * (1 - math.exp(-2 * theta * t))
    return mean, var


x = torch.full((n_paths,), x0, dtype=torch.float64)
print("   t    sample mean  exact mean   sample var  exact var")
for k in range(round(T / dt) + 1):
    t = k * dt
    if k in (0, 10, 25, 50, 100, 400):
        m, v = exact_moments(t)
        print(f"{t:5.2f}   {x.mean().item():+.4f}     {m:+.4f}     {x.var().item():.4f}     {v:.4f}")
    x = x + theta * (mu - x) * dt + sigma * math.sqrt(dt) * torch.randn_like(x)

# ── The law at time t is Gaussian: compare quantiles ────────────────────────
m, v = exact_moments(T)
normal = torch.distributions.Normal(m, math.sqrt(v))
print("\nquantiles at T:  q     sample    N(mean, var)")
for q in (0.05, 0.5, 0.95):
    print(f"              {q:4.2f}   {torch.quantile(x.float(), q).item():+.4f}   {normal.icdf(torch.tensor(q)).item():+.4f}")

   t    sample mean  exact mean   sample var  exact var
 0.00   +3.0000     +3.0000     0.0000     0.0000
 0.20   +2.3516     +2.3599     0.1042     0.1017
 0.50   +1.6331     +1.6464     0.1889     0.1863
 1.00   +0.8892     +0.9036     0.2446     0.2425
 2.00   +0.2625     +0.2722     0.2673     0.2645


 8.00   +0.0016     +0.0002     0.2681     0.2667

quantiles at T:  q     sample    N(mean, var)
              0.05   -0.8523   -0.8492
              0.50   +0.0048   +0.0002
              0.95   +0.8536   +0.8496
